**Running this on Google Colab.** The first code cell fetches this entry's files and installs the packages it was run with, pinned for Colab's Python version. The results in the registry come from the entry's own environment ([requirements.lock.txt](requirements.lock.txt), Python 3.13); a different Python or platform can change the last digits of some numbers.

In [ ]:
# On Google Colab: fetch this entry's files and install its pinned packages. Elsewhere this cell does nothing.
import sys
if "google.colab" in sys.modules:
    import os, subprocess
    if not os.path.isdir("/content/encore"):
        subprocess.run(["git", "clone", "-q", "--depth", "1", "--branch", "colab", "--filter=blob:none", "--sparse",
                        "https://github.com/encorescience/encore-registry", "/content/encore"], check=True)
        subprocess.run(["git", "-C", "/content/encore", "sparse-checkout", "set", "entries/kim-2025"], check=True)
    os.chdir("/content/encore/entries/kim-2025")
    sys.path.insert(0, os.getcwd())
    version = f"{sys.version_info.major}.{sys.version_info.minor}"
    lock = f"requirements.colab-{version}.txt"
    if not os.path.exists(lock):
        print(f"No lock for Python {version}: installing the entry's unpinned requirements (requirements.in).")
        lock = "requirements.in"
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "uv"], check=True)
    subprocess.run(["uv", "pip", "install", "--system", "-q", "-r", lock], check=True)
    print("Installed. If Colab asks to restart the session, restart it and run the notebook from the top again.")


# Reproducing Kim & Shin (2025) on its main dataset: Allen Visual Coding Neuropixels natural scenes

Paper: Kim, J. H. & Shin, H. (2025), bioRxiv doi:10.1101/2025.06.30.662469 (July 2025 preprint, the text the claims file was written from).

The registry tests this paper on dandiset 000248, which the paper uses only for pictures (first notebook). Its findings rest on the Allen Brain Observatory Visual Coding Neuropixels data, "Brain Observatory 1.1" sessions, which the paper reads through AllenSDK without naming a release. These claims are `data: elsewhere`. Here they are tested on the same sessions as deposited on DANDI: dandiset 000021, "Allen Institute - Visual Coding - Neuropixels (Brain Observatory 1.1 Stimulus Set)", version 0.251116.2246 (doi:10.48324/dandi.000021/0.251116.2246), the only published version, streamed from the archive. The 32 session files it holds match the paper's 32 Brain Observatory 1.1 sessions in number.

Units: those whose peak channel lies in the area (VISp for V1; VISl, VISrl, VISal, VISpm, VISam for LM, RL, AL, PM, AM), that pass the Allen quality criteria of the Methods (isi_violations < 0.5, amplitude_cutoff < 0.1, presence_ratio > 0.9) and whose `quality` label is "good". The label is an assumption: the Methods name only the three criteria, but the paper read the data through AllenSDK, whose default units table also keeps only units labelled "good" (`filter_by_validity`, allensdk/brain_observatory/ecephys/probes.py, lines 128 to 129, at commit a56bd8ea7613a812334602b64746cdd30c325a96), and the paper's unit counts match only with it. It is applied in every analysis; the Methods' three criteria alone are shown as a variant for the unit counts, the slope fits, behavioural state, higher visual areas and the movie data. The measures under slope manipulation were run with the Methods' criteria alone in the first round of this attempt and are not repeated here, for lack of time. Counts are spikes in the 250 ms from each natural scenes onset (Methods). Sessions with more than 10% of trials silent in all V1 units are excluded (Methods).

| Claim | Tested here |
|---|---|
| data-v1-units, fig1b-across-neuron-slope-near-1, fig1b-across-stimulus-slope, figs3-higher-visual-areas | yes |
| figs4ab-noise-correlation-under-manipulation, fig1d-dimensionality-falls-with-slope, figs5c-space-occupancy-peak | yes |
| fig2a-overlap-rises-with-slope, fig2c-continuity-from-slope-1 | yes |
| fig3bc-geodesic-orthogonal-correlation, fig3d-correlation-peaks-near-1, fig3e-local-alignment-peaks-near-1, fig3f-global-alignment-peaks-near-1 | yes |
| fig4a-overlap-consistency-within-mice, fig4c-rsm-similarity-within-mice, fig4d-overlap-consistency-across-mice, fig4e-rsm-similarity-across-mice, figs6-higher-areas-consistency | yes |
| figs2-behavioral-state | yes (pupil in the sessions with eye tracking) |
| figs5a-movie-slope-near-1 | yes, on the Functional Connectivity sessions of dandiset 000022 |
| fig2b-decoding-falls-with-slope, figs4c-shuffling-alters-dependence | in the third notebook, `reproduce_kim_2025_decoding.ipynb` (decoding at as-is and slopes 0 and 1 over the 30 sessions, and at slope 2 in four sessions) |
| figs1-gain-variance, fig2d-tangent-angle-rises-below-1 | blocked: the paper's procedures (negative binomial fits; tangent space angle) were not ported within the time of this attempt's runs; the data they need are loaded here |

Seeds: each random result uses the seed of its claim, `zlib.crc32(b"kim-2025/<claim id>")`, and the RSM-based results are repeated under the seed plus 1000, 2000, 3000 and 4000.

In [ ]:
import os
os.environ["OMP_NUM_THREADS"] = "1"
os.environ["OPENBLAS_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"
import time, zlib, itertools, resource
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from concurrent.futures import ProcessPoolExecutor
import multiprocessing as mp
from scipy.stats import mannwhitneyu, kruskal, spearmanr

import kim2025 as K

def wilcoxon(x, y=None, **kw):
    """scipy's Wilcoxon signed-rank test, with p = 1 when every difference is zero."""
    from scipy.stats import wilcoxon as _w
    d = np.asarray(x) - (0 if y is None else np.asarray(y))
    if np.all(d == 0):
        return type("R", (), {"pvalue": 1.0, "statistic": 0.0})()
    return _w(x, y, **kw) if y is not None else _w(x, **kw)

T0 = time.time()
pd.set_option("display.width", 220)
SLOPES = [float(s) for s in K.TARGET_SLOPES]
def seed(claim):
    return zlib.crc32(f"kim-2025/{claim}".encode())
paths = [p for p in K.asset_paths(K.ABO_ID, K.ABO_VERSION) if "probe" not in p]
print(f"dandiset {K.ABO_ID} version {K.ABO_VERSION}: {len(paths)} session files")

## Data check

In [ ]:
with ProcessPoolExecutor(4, mp_context=mp.get_context("fork")) as ex:
    SQ = list(ex.map(K.abo_session, paths))     # units by the Methods' three criteria
S = [K.restrict(s, s["good"]) for s in SQ]      # main reading: also labelled "good", as AllenSDK serves them
print(f"loaded in {(time.time() - T0) / 60:.1f} min; peak memory {resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 1e6:.2f} GB")
rows = []
for i, (s, sq) in enumerate(zip(S, SQ)):
    v1 = s["area"] == "V1"
    lab = s["labels"]
    _, n = np.unique(lab, return_counts=True)
    rows.append(dict(index=i, session=s["info"]["path"].split("/")[0], trials=s["info"]["trials"], stimuli=len(n),
                     repeats_min=n.min(), repeats_max=n.max(),
                     **{f"{a}_q": int((sq["area"] == a).sum()) for a in K.ABO_AREAS},
                     **{f"{a}_g": int((s["area"] == a).sum()) for a in K.ABO_AREAS},
                     V1_all=s["info"]["area_units_all"]["V1"],
                     silent_V1=round(K.silent_fraction(s["X"][:, v1]), 4),
                     silent_V1_q=round(K.silent_fraction(sq["X"][:, sq["area"] == "V1"]), 4),
                     speed_mean=round(float(np.nanmean(s["speed"])), 2), eye=s["info"]["eye_tracking"]))
tab = pd.DataFrame(rows).set_index("index")
tab["included"] = tab.silent_V1 <= K.SILENT_TRIAL_LIMIT
print("columns: <area>_q units by the Methods' three criteria, <area>_g also labelled good (main); silent_V1 with the good units, silent_V1_q with the Methods units")
print(tab.to_string())
print("frame codes:", np.unique(S[1]["labels"])[[0, 1, -1]], "(-1 is the gray image)")
print("running speed unit:", {s["info"]["running_unit"] for s in S}, "; trial-mean speed range",
      f"{min(np.nanmin(s['speed']) for s in S):.1f} to {max(np.nanmax(s['speed']) for s in S):.1f}")
allc = np.concatenate([s["X"].ravel() for s in S])
print(f"spike counts per 250 ms: min {allc.min():.0f}, max {allc.max():.0f}, mean {allc.mean():.3f}")
keep = [i for i in tab.index if tab.loc[i, "included"]]
print("excluded sessions:", [i for i in tab.index if i not in keep], "; included:", len(keep),
      "; the same under the Methods units:", list(tab.index[tab.silent_V1_q <= K.SILENT_TRIAL_LIMIT]) == keep)
ses_id = {i: int(S[i]["info"]["path"].split("_ses-")[1].split(".")[0]) for i in tab.index}
by_id = sorted(tab.index, key=ses_id.get)
print("excluded sessions by position in ascending session id (AllenSDK's order, 0-based):", {i: by_id.index(i) for i in tab.index if i not in keep})
print("duplicate check: distinct V1 count vectors", len({tuple(S[i]["X"][:, S[i]["area"] == "V1"].sum(0)) for i in tab.index}), "of", len(S))

Clock check: the image identity is decoded from V1 counts in the 250 ms after onset and in a window shifted 2 s later (several presentations later in the random sequence), in one session.

In [ ]:
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.model_selection import StratifiedKFold, cross_val_score
i0 = keep[0]
f = K.open_nwb(S[i0]["info"]["path"], K.ABO_ID, K.ABO_VERSION)
u = K.units_table(f)
rows_v1 = np.where((u.location == "VISp").to_numpy() & u.passes_quality.to_numpy())[0]
g = f["intervals/natural_scenes_presentations"]
fr, on = g["frame"][:].astype(int), g["start_time"][:]
for shift in (0.0, 2.0):
    C = K.spike_counts(f, rows_v1, on + shift, 0.25)
    acc = cross_val_score(LinearDiscriminantAnalysis(), C, fr, cv=StratifiedKFold(5, shuffle=False)).mean()
    print(f"session {i0}, window shifted {shift:+.1f} s: 119-way accuracy {acc:.3f} (chance {1/119:.4f})")
f.close()

## data-v1-units

> The main analysis uses V1 units from 30 sessions of 30 mice, 64 +- 25 units per session (14 to 110), 1920 units in all, with 119 stimuli per session (3570 stimulus fits).

In [ ]:
for label, SS in (("main: Methods criteria and the label 'good'", S), ("variant: Methods criteria alone", SQ)):
    nv1 = pd.Series({i: int((SS[i]["area"] == "V1").sum()) for i in keep})
    print(f"{label}: sessions {len(keep)} (of {len(SS)}); V1 units per session mean {nv1.mean():.1f}, SD {nv1.std(ddof=1):.1f} "
          f"(ddof 1; ddof 0 {nv1.std(ddof=0):.1f}), range {nv1.min()} to {nv1.max()}, total {nv1.sum()}")
print(f"stimuli per session {set(tab.loc[keep, 'stimuli'])}; fits {119 * len(keep)}")
print("mice: one session per subject folder,", tab.loc[keep, "session"].nunique(), "distinct subjects")


AllenSDK, through which the paper read these data, keeps only units whose `quality` label is "good" in its default units table (`filter_by_validity=True`; allensdk/brain_observatory/ecephys/probes.py, lines 128 to 129, at commit a56bd8ea7613a812334602b64746cdd30c325a96), on top of the three criteria the Methods name. The NWB files hold all units with the label. This reading is the main one in every analysis of this notebook; the Methods' criteria alone are a variant.

**Result: data-v1-units reproduced, resting on the AllenSDK reading of the unit selection.** Two of 32 sessions have more than 10% of trials silent in all V1 units (indices 0 and 9 here, the first and seventh by session id, which is how AllenSDK orders them, as the data check prints; the code drops indices 0 and 6), under either unit reading, leaving 30 sessions of 30 mice, each with 119 stimuli (3570 fits). With the label "good" there are 66.9 +- 25.5 V1 units per session (14 to 112), 2008 in all, every value within tolerance. With the Methods' three criteria alone, 71.1 +- 27.1 (14 to 113), 2133 in all: the mean and the total are outside their tolerances (57.6 to 70.4; 1728 to 2112).

## fig1b-across-neuron-slope-near-1 and fig1b-across-stimulus-slope

In [ ]:
fits = {i: K.fit_slopes(S[i]["X"][:, S[i]["area"] == "V1"], S[i]["labels"]) for i in keep}
an = pd.concat([fits[i].assign(session=i) for i in keep])
print(f"across-neuron slopes (main, good units): n {len(an)}, mean {an.slope.mean():.4f}, SEM {an.slope.sem():.4f}, SD {an.slope.std(ddof=1):.4f}, "
      f"median {an.slope.median():.4f}, range {an.slope.min():.3f} to {an.slope.max():.3f}")
print("per-session mean slope range:", round(an.groupby('session').slope.mean().min(), 3), "to", round(an.groupby('session').slope.mean().max(), 3))
print("neurons per fit (mean > 0): median", an.n.median(), ", min", an.n.min())
an_q = pd.concat([K.fit_slopes(SQ[i]["X"][:, SQ[i]["area"] == "V1"], SQ[i]["labels"]) for i in keep])
print(f"variant, Methods criteria alone: n {len(an_q)}, mean {an_q.slope.mean():.4f}, SEM {an_q.slope.sem():.4f}, SD {an_q.slope.std(ddof=1):.4f}")


**Result: fig1b-across-neuron-slope-near-1 reproduced, resting on the AllenSDK reading of the unit selection.** With the good units the 3570 across-neuron slopes have mean 1.0489 (SEM 0.0011) and SD 0.0665, within 1.048 +- 0.001 and 0.07 +- 0.007; they lie between 0.605 and 1.381. With the Methods' three criteria alone the mean is 1.0552 (SEM 0.0011), outside the tolerance, and the SD 0.0655 within it.

In [ ]:
ast = np.concatenate([K.across_stimulus_slopes(S[i]["X"][:, S[i]["area"] == "V1"], S[i]["labels"]) for i in keep])
dropped = int(np.isnan(ast).sum()); ast = ast[~np.isnan(ast)]
mw = mannwhitneyu(ast, an.slope, alternative="two-sided")
print(f"across-stimulus slopes: n {len(ast)} ({dropped} neurons without 2 stimuli of nonzero mean dropped), mean {ast.mean():.4f}, SD {ast.std(ddof=1):.4f}, median {np.median(ast):.4f}")
print(f"across-neuron: mean {an.slope.mean():.4f}, median {an.slope.median():.4f}; rank-sum (Mann-Whitney U, two-sided) U = {mw.statistic:.0f}, p = {mw.pvalue:.3g}")
ast_q = np.concatenate([K.across_stimulus_slopes(SQ[i]["X"][:, SQ[i]["area"] == "V1"], SQ[i]["labels"]) for i in keep])
ast_q = ast_q[~np.isnan(ast_q)]
print(f"variant, Methods criteria alone: across-stimulus n {len(ast_q)}, mean {ast_q.mean():.4f}, SD {ast_q.std(ddof=1):.4f}, median {np.median(ast_q):.4f}; "
      f"across-neuron mean {an_q.slope.mean():.4f}, median {an_q.slope.median():.4f}; Mann-Whitney p = {mannwhitneyu(ast_q, an_q.slope).pvalue:.3g}")
fig, ax = plt.subplots(figsize=(5, 3))
ax.hist(an.slope, bins=np.linspace(0, 3, 61), alpha=0.6, density=True, label="across neurons")
ax.hist(ast, bins=np.linspace(0, 3, 61), alpha=0.6, density=True, label="across stimuli"); ax.legend(); ax.set_xlabel("slope")
print("points outside 0 to 3:", int((an.slope < 0).sum() + (an.slope > 3).sum()), "across-neuron,", int((ast < 0).sum() + (ast > 3).sum()), "across-stimulus")
plt.show()

**Result: fig1b-across-stimulus-slope reproduced.** The 2008 across-stimulus slopes of the good units have mean 1.135 and SD 0.207 (expected 1.13 +- 0.21 and 0.21 +- 0.021), larger than the across-neuron slopes in mean (1.135 against 1.049) and median (1.151 against 1.052), two-sided rank-sum p = 7e-117. With the Methods' criteria alone (2133 units), mean 1.134 and SD 0.206, also larger in mean and median, p = 5e-105.

## figs2-behavioral-state

> The across-neuron slope is slightly but significantly modulated by behavioral state, smallest when the mouse runs and when its pupils are dilated, while staying near 1.

Locomotion: trial mean running speed >= 1 cm/s is moving; M1 per session, stimulus and state, pooled; stimulus and state cells with fewer than 2 trials cannot give a variance and are dropped. Pupil: trial mean of the blink-filtered pupil area (`processing/filtered_gaze_mapping/pupil_area`), present in the sessions marked `eye` in the data check; tertiles over the trials of all sessions pooled (code, fit_slope_histogram.ipynb cell 67), per-session tertiles as a variant; Kruskal-Wallis across the three groups of slopes.

In [ ]:
def state_slope(X, m):
    Y = X[m]; mu, v = Y.mean(0), Y.var(0, ddof=1); ok = (mu > 0) & (v > 0)
    return np.polyfit(np.log10(mu[ok]), np.log10(v[ok]), 1)[0] if ok.sum() >= 2 else None

pup = {i: K.abo_pupil(S[i]["info"]["path"], K.abo_onsets(S[i]["info"]["path"])) for i in keep}
pup = {i: v for i, v in pup.items() if v is not None}
allp = np.concatenate([v for v in pup.values()])
edges = np.nanpercentile(allp, [100 / 3, 200 / 3])
print(f"pupil: {len(pup)} included sessions with eye tracking; trials without a pupil value {np.isnan(allp).mean():.3f}; "
      f"pooled tertile edges {edges[0]:.4f} and {edges[1]:.4f}; range {np.nanmin(allp):.4g} to {np.nanmax(allp):.4g} (labelled pixels^2)")
print("fraction of trials moving (trial mean speed >= 1 cm/s):", round(float(np.mean(np.concatenate([S[i]["speed"] >= 1 for i in keep]))), 3))

for reading, SS in (("main, good units", S), ("variant, Methods criteria alone", SQ)):
    print(f"=== {reading}")
    st_rows = []
    for i in keep:
        X, lab, sp = SS[i]["X"][:, SS[i]["area"] == "V1"], SS[i]["labels"], SS[i]["speed"]
        for state, sel in (("moving", sp >= 1), ("stationary", sp < 1)):
            for c in np.unique(lab):
                m = (lab == c) & sel
                if m.sum() >= 2 and (b := state_slope(X, m)) is not None:
                    st_rows.append((i, state, c, b, m.sum()))
    bs = pd.DataFrame(st_rows, columns=["session", "state", "stim", "slope", "trials"])
    print(bs.groupby("state").slope.describe().round(4).to_string())
    mv, stn = bs[bs.state == "moving"].slope, bs[bs.state == "stationary"].slope
    print(f"moving vs stationary: medians {mv.median():.4f} vs {stn.median():.4f}; Mann-Whitney two-sided p = {mannwhitneyu(mv, stn).pvalue:.3g}; "
          f"sessions with moving fits {bs[bs.state == 'moving'].session.nunique()}")
    for label, per_session in (("pooled tertiles (code)", False), ("per-session tertiles (variant)", True)):
        pr = []
        for i, pv in pup.items():
            X, lab = SS[i]["X"][:, SS[i]["area"] == "V1"], SS[i]["labels"]
            e = np.nanpercentile(pv, [100 / 3, 200 / 3]) if per_session else edges
            tert = np.where(np.isnan(pv), -1, np.digitize(pv, e))
            for t_ in range(3):
                for c in np.unique(lab):
                    m = (lab == c) & (tert == t_)
                    if m.sum() >= 2 and (b := state_slope(X, m)) is not None:
                        pr.append((i, t_, b))
        pdf = pd.DataFrame(pr, columns=["session", "tertile", "slope"])
        meds = pdf.groupby("tertile").slope.median()
        kw = kruskal(*[pdf[pdf.tertile == t_].slope for t_ in range(3)])
        print(f"{label}: fits per tertile {pdf.groupby('tertile').size().to_dict()}, median slope per tertile {meds.round(4).to_dict()}, "
              f"largest tertile lowest: {bool(meds.idxmin() == 2)}, Kruskal-Wallis p = {kw.pvalue:.3g}, medians in 0.9-1.1: {bool(meds.between(0.9, 1.1).all())}")
    print("per-session median slope, moving minus stationary:",
          (bs.groupby(["session", "state"]).slope.median().unstack().dropna().pipe(lambda d: d.moving - d.stationary)).describe().round(4).to_dict())


**Result: reproduced.** With the good units, moving trials (44.6% of trials) give lower slopes than stationary ones (medians 0.991 against 1.061, rank-sum p = 2e-307). Over the 25 included sessions with eye tracking, the largest pupil tertile has the lowest median slope (1.029, against 1.070 and 1.046), Kruskal-Wallis p = 1e-48 with pooled tertiles (per-session tertiles, 1.034 against 1.067 and 1.046, p = 5e-42). Every state's median slope lies between 0.9 and 1.1. With the Methods' criteria alone the same holds (medians 0.999 against 1.067; pupil tertiles 1.077, 1.053, 1.039). Five included sessions have no eye tracking, and 10.5% of trials in the others have no pupil value.

## figs3-higher-visual-areas

> The across-neuron slope is concentrated around 1 in the five higher visual areas as in V1.

M1 per area in every session with at least two quality units in the area (a line needs two neurons), over all 32 sessions and over the 30 included sessions, and, as a variant, over the sessions of all 32 that have no more than 10% of trials silent in all units of the area (the Methods exclude "2 sessions for V1 and 1 session for RL" by this rule; Methods, Main dataset and visual stimuli).

In [ ]:
for reading, SS in (("good units (main)", S), ("Methods criteria alone (variant)", SQ)):
    for label, idx in (("all 32 sessions", list(tab.index)), ("30 included sessions", keep), ("all 32 sessions, area-specific silence exclusion", list(tab.index))):
        out = {}
        for a in K.ABO_AREAS:
            sl, ns = [], 0
            for i in idx:
                m = SS[i]["area"] == a
                if label.endswith("exclusion") and m.sum() and K.silent_fraction(SS[i]["X"][:, m]) > K.SILENT_TRIAL_LIMIT:
                    continue
                if m.sum() >= 2:
                    ns += 1
                    sl.append(K.fit_slopes(SS[i]["X"][:, m], SS[i]["labels"]).slope.to_numpy())
            sl = np.concatenate(sl)
            out[a] = dict(sessions=ns, sessions_any_unit=int(sum((SS[i]["area"] == a).sum() > 0 for i in idx)), fits=len(sl),
                          median=round(float(np.median(sl)), 4), mean=round(float(sl.mean()), 4),
                          within_0_9_1_1=bool(0.9 <= np.median(sl) <= 1.1))
        print(f"{reading}, {label}"); print(pd.DataFrame(out).T.to_string())


**Result: reproduced.** With the good units, over all 32 sessions, the sessions with at least two units are LM 24, RL 28, AL 24, PM 20 and AM 26, each within tolerance of 24, 27, 24, 20 and 26, and the median slopes are 1.056, 1.047, 1.045, 1.002 and 1.024, all between 0.9 and 1.1 (V1 1.057). The Methods' criteria alone give the same session counts, and over the 30 included sessions (22, 26, 23, 18 and 25) the medians again lie between 0.9 and 1.1 under both readings. RL has one session more than the paper's 27: with the Methods' area-specific exclusion of sessions with more than 10% of trials silent in all units of the area, which removes one RL session (and the two V1 sessions), the counts over all 32 sessions are 24, 27, 24, 20 and 26, the paper's exactly, under both unit readings, with medians 1.056, 1.044, 1.045, 1.002 and 1.024 (good units).

## Measures under slope manipulation

For every included session, V1 units, the as-is data and the 21 target slopes 0 to 2, `K.session_measures` computes the effective dimensionality, the PC1 fraction, pairwise noise correlations, the overlap matrices (Methods and code thresholds), occupied voxels, geodesic distances and orthogonal variances, local and global alignment, the RSM, and the half-split consistencies; and, for LM and PM, the RSM and its half-split consistency. Sessions run in 4 processes.

In [ ]:
SEEDS = {k: seed(k) for k in ["fig4e-rsm-similarity-across-mice", "fig4c-rsm-similarity-within-mice",
                              "fig4a-overlap-consistency-within-mice", "figs6-higher-areas-consistency"]}
def job(i, seeds=SEEDS, halves=True):
    s = S[i]
    areas = {a: s["X"][:, s["area"] == a] for a in ("LM", "PM") if (s["area"] == a).sum() >= 2}
    return K.session_measures(s["X"][:, s["area"] == "V1"], s["labels"], seeds, areas=areas, do_halves=halves)
t = time.time()
with ProcessPoolExecutor(4, mp_context=mp.get_context("fork")) as ex:
    M = dict(zip(keep, ex.map(job, keep)))
print(f"measures for {len(M)} sessions in {(time.time() - t) / 60:.1f} min")
CONDS = ["as-is"] + SLOPES

def curve(key, f=lambda v: v):
    return pd.DataFrame({i: {c: f(M[i][key][c]) for c in CONDS} for i in M}).T

def o1(df, label, n_label="sessions"):
    m = df[SLOPES].mean()
    peak = m.idxmax()
    p0, p2 = wilcoxon(df[0.0], df["as-is"]).pvalue, wilcoxon(df[2.0], df["as-is"]).pvalue
    lower = bool(m[0.0] < df["as-is"].mean() and m[2.0] < df["as-is"].mean())
    res = dict(peak_slope=peak, peak_mean=round(m.max(), 4), mean_asis=round(df["as-is"].mean(), 4), mean_slope0=round(m[0.0], 4),
               mean_slope1=round(m[1.0], 4), mean_slope2=round(m[2.0], 4), p_slope0=float(f"{p0:.3g}"), p_slope2=float(f"{p2:.3g}"),
               ends_lower=lower, reading_a=bool(0.8 <= peak <= 1.2 and lower and p0 < 0.05 and p2 < 0.05),
               reading_b=bool(0.7 <= peak <= 1.3 and lower and p0 < 0.05 and p2 < 0.05))
    print(f"{label} (n = {len(df)} {n_label}):", res)
    print("   curve:", " ".join(f"{c}:{v:.3f}" for c, v in m.items()))
    return res

def o2(df, direction, label):
    m = df[SLOPES].mean()
    a0, a1, a2, asis = m[0.0], m[1.0], m[2.0], df["as-is"].mean()
    order = (a2 < a1 < a0) if direction == "decrease" else (a2 > a1 > a0)
    p0, p2 = wilcoxon(df[0.0], df["as-is"]).pvalue, wilcoxon(df[2.0], df["as-is"]).pvalue
    steps = np.diff(m.to_numpy())
    strict = bool(np.all(steps < 0)) if direction == "decrease" else bool(np.all(steps > 0))
    res = dict(mean_asis=round(asis, 4), mean_slope0=round(a0, 4), mean_slope1=round(a1, 4), mean_slope2=round(a2, 4),
               order=order, p_slope0=float(f"{p0:.3g}"), p_slope2=float(f"{p2:.3g}"), strict_every_step=strict,
               holds=bool(order and p0 < 0.05 and p2 < 0.05))
    print(f"{label} (n = {len(df)} sessions):", res)
    print("   curve:", " ".join(f"{c}:{v:.3f}" for c, v in m.items()))
    return res

## figs4ab-noise-correlation-under-manipulation

> Rescaling residuals to change the slope leaves pairwise noise correlations unchanged but changes population-wise noise correlation.

In [ ]:
ncd = pd.DataFrame({i: M[i]["nc_maxdiff"] for i in M}).T
print("largest |noise correlation change| over sessions, stimuli and pairs, at slopes 0, 1, 2:", ncd.max().to_dict())
pc = curve("pc1")
for sl in (0.0, 2.0):
    frac = [np.mean(np.abs(M[i]["pc1"][sl] - M[i]["pc1"]["as-is"]) > 1e-9) for i in M]
    print(f"slope {sl}: fraction of stimuli whose PC1 variance fraction differs from as-is, per session min {min(frac):.3f}, mean {np.mean(frac):.3f}; "
          f"pooled mean PC1 fraction {np.mean([np.nanmean(M[i]['pc1'][sl]) for i in M]):.4f} vs as-is {np.mean([np.nanmean(M[i]['pc1']['as-is']) for i in M]):.4f}")
ex7 = keep[7] if len(keep) > 7 else keep[0]
print(f"example (session index {ex7}): mean PC1 fraction as-is {np.nanmean(M[ex7]['pc1']['as-is']):.4f}, slope 0 {np.nanmean(M[ex7]['pc1'][0.0]):.4f}, slope 2 {np.nanmean(M[ex7]['pc1'][2.0]):.4f}")

**Result: reproduced.** Over the 30 sessions, every stimulus's pairwise noise correlations at slopes 0, 1 and 2 equal the as-is values to within 8.0e-15, and the fraction of variance on the first principal component differs from as-is for every stimulus at slopes 0 and 2 (pooled mean 0.161 at slope 0 and 0.507 at slope 2, against 0.335 as-is).

## fig1d-dimensionality-falls-with-slope

In [ ]:
dim = curve("dim", np.mean)
r_dim = o2(dim, "decrease", "effective dimensionality")

**Result: reproduced.** The mean effective dimensionality falls at every 0.1 step from 0.292 at slope 0 to 0.129 at slope 1 and 0.065 at slope 2 (as-is 0.125); slopes 0 and 2 differ from as-is by the Wilcoxon signed-rank test (p = 1.9e-09 each).

## figs5c-space-occupancy-peak

In [ ]:
vox = curve("vox").astype(float)
r_vox = o1(vox, "occupied unit voxels")
print("sessions where the as-is count equals the number of trials (every trial in its own voxel):",
      int(sum(vox.loc[i, 'as-is'] == len(S[i]['labels']) for i in M)), "of", len(M))
print("sessions whose maximum over slopes is reached at more than one slope:", int(sum((vox.loc[i, SLOPES] == vox.loc[i, SLOPES].max()).sum() > 1 for i in M)))

**Result: reproduced.** The mean count of occupied unit voxels peaks at slope 0.8 (5949.9), within both windows, and is lower at slope 0 (5728.6) and slope 2 (379.1) than as-is (5949.9), with p = 1.7e-06 for each. The measure is at its ceiling near the peak: in 28 of 30 sessions every as-is trial occupies a voxel of its own, and in 28 sessions the maximum is shared by more than one slope, so the peak at 0.8 marks the middle of a plateau that runs from about 0.6 to 1.0 (means 5946.5 to 5949.9). The fall at high slopes comes from points that leave the as-is box and are dropped, as compute_percent_filled does (box_and_overlap.py, lines 107 to 111 at ac3300b), which also anchors the grid at the minimum of the points kept (line 122), as here.

## fig2a-overlap-rises-with-slope and fig2c-continuity-from-slope-1

In [ ]:
ov = curve("O_m", K.mean_offdiag)
r_ov = o2(ov, "increase", "overlap, distinct pairs, Methods threshold (main)")
ov_code = curve("O_c", lambda O: float(O.mean()))
r_ov_code = o2(ov_code, "increase", "overlap, code variant (code threshold, full matrix with diagonal)")
ov_c2 = curve("O_c", K.mean_offdiag)
o2(ov_c2, "increase", "overlap, code threshold, distinct pairs")

**Result: fig2a-overlap-rises-with-slope reproduced.** The mean overlap over ordered pairs of distinct stimuli rises at every 0.1 step from 0.065 at slope 0 to 0.279 at slope 1 and 0.422 at slope 2 (as-is 0.288), with p = 1.9e-09 for slopes 0 and 2 against as-is. The code's variant (its threshold, the full matrix with the diagonal) gives 0.049, 0.225 and 0.318 (p = 1.9e-09 and 4.7e-07), also rising at every step.

In [ ]:
scc = curve("O_m", K.largest_scc).astype(float)
scc_c = curve("O_c", K.largest_scc).astype(float)
for name, df in (("Methods threshold", scc), ("code threshold", scc_c)):
    m = df[SLOPES].mean()
    full = [c for c in SLOPES if m[c] >= 119]
    first = full[0] if full else None
    try:
        p0 = wilcoxon(df[0.0], df["as-is"]).pvalue
    except ValueError:
        p0 = float("nan")
    print(f"{name}: mean largest SCC size by slope", " ".join(f"{c}:{v:.1f}" for c, v in m.items()), f"| as-is {df['as-is'].mean():.1f}")
    print(f"   first slope with mean 119: {first}; reading a (0.8-1.2): {first is not None and 0.8 <= first <= 1.2}; reading b (0.7-1.3): {first is not None and 0.7 <= first <= 1.3}; "
          f"slope 0 mean {m[0.0]:.1f}, Wilcoxon vs as-is p = {p0:.3g}; sessions at 119 as-is {int((df['as-is'] == 119).sum())} of {len(df)}")
    print("   sessions reaching 119 by slope:", " ".join(f"{c}:{int((df[c] == 119).sum())}" for c in SLOPES))

**Result: fig2c-continuity-from-slope-1 reproduced.** The mean size of the largest strongly connected component first reaches 119 at slope 1.1 (Methods threshold; 1.2 with the code's threshold), inside both windows; at slope 0 it is 108.1 (104.3) and differs from as-is (118.9, 29 of 30 sessions complete) with p = 5.8e-05 (4.0e-05). The number of sessions whose component covers all stimuli grows from 9 at slope 0 to 29 at slope 1 and 30 from 1.1.

## fig3bc-geodesic-orthogonal-correlation, fig3d, fig3e, fig3f

In [ ]:
geo = curve("geo_rho")
rho = geo["as-is"]
w = wilcoxon(rho, alternative="greater")
print(f"as-is Spearman rho per session: mean {rho.mean():.4f}, range {rho.min():.4f} to {rho.max():.4f}, positive in {(rho > 0).sum()} of {len(rho)}; "
      f"one-sided Wilcoxon against 0 p = {w.pvalue:.3g}; pairs per session {int(np.isfinite(K._offdiag(M[keep[0]]['asis_ortho'])).sum())}")
print("sessions with non-finite geodesics:", int(sum((~np.isfinite(M[i]["asis_geo"])).any() for i in M)))

**Result: fig3bc-geodesic-orthogonal-correlation reproduced.** The as-is Spearman correlation between inter-centroid geodesic distance and orthogonal variance over the 14042 ordered pairs is positive in all 30 sessions (mean 0.325, 0.028 to 0.564), and the one-sided Wilcoxon test against 0 gives p = 9.3e-10.

In [ ]:
r_geo = o1(geo, "geodesic-orthogonal variance Spearman")

**Result: fig3d-correlation-peaks-near-1 reproduced.** The mean correlation peaks at slope 1.2 (0.332), inside both windows; it is lower at slope 0 (0.228) and slope 2 (0.307) than as-is (0.325), with p = 3.2e-07 and 0.0093.

In [ ]:
r_loc = o1(curve("local"), "local alignment (as-is neighbour, code)")
o1(curve("local_slope_nn"), "local alignment (variant, neighbour from the slope-changed data)")

**Result: fig3e-local-alignment-peaks-near-1 reproduced.** Mean local alignment peaks at slope 1.0 (0.823); slope 0 (0.727) and slope 2 (0.793) are lower than as-is (0.822), p = 1.4e-06 and 0.0011. With the neighbour taken from the slope-changed data (variant) the peak is also at 1.0.

In [ ]:
r_glo = o1(curve("global"), "global alignment")

**Result: fig3f-global-alignment-peaks-near-1 reproduced.** Mean global alignment peaks at slope 1.0 (0.832); slope 0 (0.698) and slope 2 (0.796) are lower than as-is (0.832), p = 1.6e-07 and 0.00021.

## fig4a, fig4c, fig4d, fig4e, figs6

In [ ]:
r_4a = o1(curve("ov_half"), "overlap consistency within session, full matrices (code, main)")
o1(curve("ov_half_off"), "overlap consistency within session, off-diagonal (variant)")

**Result: fig4a-overlap-consistency-within-mice reproduced.** The mean Spearman correlation between the overlap matrices of two random halves of the neurons peaks at slope 0.9 (0.651, with 0.648 at 1.0); slope 0 (0.427) and slope 2 (0.358) are lower than as-is (0.641), p = 1.9e-09 each. The off-diagonal variant also peaks at 0.9. Only the claim's seed was run for this measure.

In [ ]:
r_4c = o1(curve("rsm_half"), "RSM similarity within session")

**Result: fig4c-rsm-similarity-within-mice reproduced.** The mean half-split RSM correlation peaks at slope 1.1 (0.435); slope 0 (0.409) and slope 2 (0.216) are lower than as-is (0.456), p = 0.0011 and 1.9e-08. The as-is value is higher than at any changed slope. Under the four further seeds the peak is at 1.1 each time (seed check at the end of `reproduce_kim_2025_decoding.ipynb`).

In [ ]:
pairs = list(itertools.combinations(keep, 2))
def across(key, f):
    return pd.DataFrame({p: {c: f(M[p[0]][key][c], M[p[1]][key][c]) for c in CONDS} for p in pairs}).T
full_sp = lambda A, B: spearmanr(A.ravel(), B.ravel()).statistic
off_sp = lambda A, B: spearmanr(K._offdiag(A), K._offdiag(B)).statistic
r_4d = o1(across("O_m", full_sp), "overlap consistency across sessions, full matrices (main)", "session pairs")
o1(across("O_m", off_sp), "overlap consistency across sessions, off-diagonal (variant)", "session pairs")

**Result: fig4d-overlap-consistency-across-mice reproduced.** Over the 435 session pairs, the mean Spearman correlation between overlap matrices peaks at slope 1.2 (0.430); slope 0 (0.253) and slope 2 (0.348) are lower than as-is (0.411), p = 2.0e-69 and 1.1e-31. The off-diagonal variant also peaks at 1.2.

In [ ]:
rsm_sp = lambda A, B: spearmanr(A.ravel(), B.ravel(), nan_policy="omit").statistic
r_4e = o1(across("rsm", rsm_sp), "RSM similarity across sessions", "session pairs")

**Result: fig4e-rsm-similarity-across-mice reproduced.** Over the 435 session pairs the mean RSM correlation peaks at slope 1.1 (0.388); slope 0 (0.348) and slope 2 (0.186) are lower than as-is (0.398), p = 1.3e-51 and 1.5e-70. The as-is value is higher than at any changed slope. Under the four further seeds the peak is at 1.1 each time (seed check at the end of `reproduce_kim_2025_decoding.ipynb`).

In [ ]:
def near_max(df, n_label):
    m, se = df[SLOPES].mean(), df[SLOPES].sem()
    peak = m.idxmax()
    a = bool(m[1.0] >= m.max() - se[peak])
    b = bool(0.8 <= peak <= 1.2)
    b2 = bool(0.7 <= peak <= 1.3)
    return dict(n=len(df), peak=peak, mean_at_1=round(m[1.0], 4), max=round(m.max(), 4), sem_at_peak=round(se[peak], 4),
                reading_a=a, reading_b_08_12=b, reading_b_07_13=b2)
s6 = {}
for a in ("LM", "PM"):
    ids = [i for i in M if a in M[i]["area_rsm_half"]]
    within = pd.DataFrame({i: M[i]["area_rsm_half"][a] for i in ids}).T
    pa = list(itertools.combinations(ids, 2))
    acr = pd.DataFrame({p: {c: rsm_sp(M[p[0]]["area_rsm"][a][c], M[p[1]]["area_rsm"][a][c]) for c in CONDS} for p in pa}).T
    s6[(a, "S6B within")] = near_max(within, "sessions")
    s6[(a, "S6C across")] = near_max(acr, "pairs")
    print(a, "within curve:", " ".join(f"{c}:{v:.3f}" for c, v in within[SLOPES].mean().items()))
    print(a, "across curve:", " ".join(f"{c}:{v:.3f}" for c, v in acr[SLOPES].mean().items()))
print(pd.DataFrame(s6).T.to_string())

**Result: figs6-higher-areas-consistency reproduced under reading (a).** For LM and PM, within sessions (S6B; 22 and 18 sessions) and across session pairs (S6C; 231 and 153 pairs), the mean at slope 1 lies within one SEM of the curve's maximum in all four panels (reading a). Under reading (b) the maximum lies in the window only across pairs (LM 1.2, PM 0.9); within sessions the curves are nearly flat from slope 0 to 1.1 and peak at 0.3 (LM, 0.262 against 0.255 at slope 1, SEM 0.023) and 0.3 (PM, 0.317 against 0.298, SEM 0.037). Within sessions the SEM is larger than the spread of the curve between slopes 0 and 1.1, so slope 0 also lies within one SEM of the maximum there, and reading (a) does not tell slope 1 from slope 0 in those two panels. Reading (a) holds in all four panels under every seed of the seed check in `reproduce_kim_2025_decoding.ipynb`.

## figs5a-movie-slope-near-1

> In the natural movie data the across-neuron log-mean against log-variance relation is also linear with slope near 1.

The paper's natural movie data are the Allen "Functional Connectivity" sessions, deposited as dandiset 000022, version 0.251116.2247 (the only published version; doi:10.48324/dandi.000022/0.251116.2247), streamed. In these files the 60-repeat block of natural movie one is `natural_movie_one_more_repeats_presentations` (a 20-repeat shuffled version is a separate table). Each repeat is cut into 75 bins of 12 frames; M1 per bin and session.

Exclusions. The Methods exclude 3 of 26 sessions "for truncated movies or no V1 units". The paper's code skips sessions 5, 14 and 43 of AllenSDK's session table, all 58 Brain Observatory 1.1 and Functional Connectivity sessions, with the comment "movie one presentation had trouble in some trials...stimtable.stimulus_name 'invalid_presentation' (session 5, 14, 43)" (extract_natural_movie_neuropixels.m, lines 19 and 101, at ac3300b). AllenSDK marks a presentation invalid when it overlaps an `invalid_times` interval tagged "stimulus" (`_mask_invalid_stimulus_presentations`, allensdk/brain_observatory/ecephys/ecephys_session.py, lines 645 to 680, at commit a56bd8ea7613a812334602b64746cdd30c325a96); here a frame counts when its onset lies in such an interval. Below, the 58 session ids of the two dandisets are put in AllenSDK's order (ascending session id) to show which sessions 5, 14 and 43 are, and each Functional Connectivity session's movie block is checked for such intervals, for incomplete repeats, for its longest repeat and for its longest gap between frames. A session is used when no movie frame falls in an invalid "stimulus" interval, every repeat is complete, and it has at least two V1 units. Units: the Methods' criteria and the label "good" (main, as for the natural scenes; the code reads AllenSDK's units table), and the Methods' criteria alone (variant).

In [ ]:
import re
all_ids = sorted([(int(re.search(r"ses-(\d+)", p_).group(1)), "Brain Observatory 1.1", p_.split("/")[0])
                  for p_ in K.asset_paths(K.ABO_ID, K.ABO_VERSION) if "probe" not in p_]
                 + [(int(re.search(r"ses-(\d+)", p_).group(1)), "Functional Connectivity", p_.split("/")[0])
                    for p_ in K.asset_paths(K.FC_ID, K.FC_VERSION) if "probe" not in p_])
print(f"{len(all_ids)} sessions in ascending session id; the code's sessions 5, 14 and 43 (1-based):")
for k in (5, 14, 43):
    print("  ", k, all_ids[k - 1])
with ProcessPoolExecutor(4, mp_context=mp.get_context("fork")) as ex:
    bo_inv = list(ex.map(K.movie_invalid_frames, paths, [K.ABO_ID] * len(paths), [K.ABO_VERSION] * len(paths)))
print("Brain Observatory 1.1 sessions (dandiset 000021) with an invalid_times interval tagged stimulus, and the natural movie one frames inside one:")
for r in bo_inv:
    if any("stimulus" in t for _, _, t in r["intervals"]):
        print("  ", r["session"], {k: v for k, v in r.items() if k.startswith("natural_movie_one")},
              [(a, b) for a, b, t in r["intervals"] if "stimulus" in t])
print("Brain Observatory 1.1 sessions with natural movie one frames inside such an interval:",
      [r["session"] for r in bo_inv if any(v > 0 for k, v in r.items() if k.startswith("natural_movie_one"))])
fc_paths = [p for p in K.asset_paths(K.FC_ID, K.FC_VERSION) if "probe" not in p]
with ProcessPoolExecutor(4, mp_context=mp.get_context("fork")) as ex:
    movie = list(ex.map(K.movie_session, fc_paths))
mvt = pd.DataFrame([dict(session=r["path"].split("/")[0], **{k: v for k, v in r.items() if k != "path"}) for _, _, _, r in movie])
mvt["valid"] = (mvt.invalid_stimulus_frames == 0) & (mvt.repeats_incomplete == 0)
print(mvt.to_string())
print("Functional Connectivity sessions with movie frames inside such an interval:", list(mvt.session[mvt.invalid_stimulus_frames > 0]))
for reading, use_good in (("main, good units", True), ("variant, Methods criteria alone", False)):
    for excl, need_valid in (("invalid movie sessions excluded (code)", True), ("no invalid exclusion", False)):
        sl, n_used = [], 0
        for (Xm, labm, goodm, r), v in zip(movie, mvt.valid):
            Xu = Xm[:, goodm] if use_good else Xm
            if Xu.shape[1] >= 2 and (v or not need_valid):
                n_used += 1
                sl.append(K.fit_slopes(Xu, labm).slope.to_numpy())
        ms = np.concatenate(sl)
        print(f"{reading}, {excl}: sessions used {n_used} of {len(movie)}; bins per session {set(len(x) for x in sl)}; fits {len(ms)}; "
              f"median {np.median(ms):.4f}, mean {ms.mean():.4f}, SD {ms.std(ddof=1):.4f}, range {ms.min():.3f} to {ms.max():.3f}; "
              f"median in 0.9-1.1: {0.9 <= np.median(ms) <= 1.1}, in 0.85-1.15: {0.85 <= np.median(ms) <= 1.15}")


**Result: reproduced, resting on the reading of the paper's code for the excluded session.** The code's sessions 5, 14 and 43 in AllenSDK's order are sub-718643564 and sub-719828686 (Brain Observatory 1.1) and sub-769319621 (Functional Connectivity), and these are the three sessions of the 58 whose files hold an `invalid_times` interval tagged "stimulus" that overlaps natural movie one (239 and 954 frames of `natural_movie_one_presentations` in the two Brain Observatory 1.1 sessions; two other Brain Observatory 1.1 sessions, sub-703279277 and sub-716813540, have such intervals that do not overlap the movie). In sub-769319621, 1497 frames of the 60-repeat movie block fall in that interval, its longest repeat lasts 30.719 s (30.025 s elsewhere, 30.042 s at most) and its longest gap between frames is 0.133 s (0.050 s at most elsewhere). No session has an incomplete repeat. Dropping sub-769319621 and the two sessions without V1 units leaves 23 of 26 sessions, as the paper says, each with 75 bins. With the good units the 1725 pooled slopes have median 1.116: outside 0.9 to 1.1 (reading 1) and inside 0.85 to 1.15 (reading 2); with the Methods' criteria alone, median 1.129, the same under both readings. Keeping sub-769319621 (24 sessions) gives medians 1.119 and 1.131.

## Summary

All claims here are `data: elsewhere`: they are tested on the Allen data as deposited in dandisets 000021 and 000022, not on dandiset 000248. Units are those passing the Methods' three criteria and labelled "good", an assumption (header) on which every outcome below rests; the Methods' criteria alone are shown for the unit counts, the slope fits, behavioural state, higher visual areas and the movie data.

| Claim | Outcome |
|---|---|
| data-v1-units | reproduced (assumed: AllenSDK's default "good" unit label; not within tolerance without it) |
| fig1b-across-neuron-slope-near-1 | reproduced (same assumption; the mean is outside tolerance without it) |
| fig1b-across-stimulus-slope | reproduced (both unit readings) |
| figs2-behavioral-state | reproduced (both unit readings) |
| figs3-higher-visual-areas | reproduced (both unit readings) |
| figs4ab-noise-correlation-under-manipulation | reproduced |
| fig1d-dimensionality-falls-with-slope | reproduced |
| figs5c-space-occupancy-peak | reproduced (the measure is at its ceiling from about 0.6 to 1.0) |
| fig2a-overlap-rises-with-slope | reproduced |
| fig2c-continuity-from-slope-1 | reproduced |
| fig3bc-geodesic-orthogonal-correlation | reproduced |
| fig3d-correlation-peaks-near-1 | reproduced |
| fig3e-local-alignment-peaks-near-1 | reproduced |
| fig3f-global-alignment-peaks-near-1 | reproduced |
| fig4a-overlap-consistency-within-mice | reproduced |
| fig4c-rsm-similarity-within-mice | reproduced |
| fig4d-overlap-consistency-across-mice | reproduced |
| fig4e-rsm-similarity-across-mice | reproduced |
| figs6-higher-areas-consistency | reproduced (reading a, which within sessions does not tell slope 1 from slope 0) |
| figs5a-movie-slope-near-1 | reproduced (assumed: the third excluded session is the one AllenSDK marks invalid, as the paper's code reads; 23 sessions; median 1.116, inside the second reading's window only) |
| fig2b-decoding-falls-with-slope | blocked, too large (run in part in `reproduce_kim_2025_decoding.ipynb`: as-is and slopes 0 and 1 in 30 sessions, slope 2 in four) |
| figs4c-shuffling-alters-dependence | reproduced (`reproduce_kim_2025_decoding.ipynb`) |
| figs1-gain-variance, fig2d-tangent-angle-rises-below-1 | blocked: the procedures were not ported within the time of this attempt's runs |

How the analysis differs from the paper's: the data are read from the DANDI copies of the Allen release with h5py, not through AllenSDK. The units are those AllenSDK serves by default (the Methods' three criteria and the label "good"); the Methods name only the three criteria. The measures under slope manipulation were not repeated here with the Methods' criteria alone. The slope fits, slope changes, overlap, geodesic distances, orthogonal variance, alignment and RSMs follow the claims file and the paper's code at commit ac3300b, including the code's as-is nearest neighbour for local alignment. The movie sessions are excluded as the paper's code excludes them, by AllenSDK's invalid presentations. The random neuron partitions and trial orders use this attempt's seeds, not the code's `np.random.seed(0)` sequence.

What was found about the data: the 32 Brain Observatory 1.1 sessions of dandiset 000021 match the paper's 32, the exclusion rule removes the two sessions that the paper's code drops (by position in ascending session id, printed in the data check; this order is AllenSDK's), the sessions per higher visual area match the paper once the Methods' area-specific exclusion removes one RL session (RL 28 without it), and the V1 unit counts and the across-neuron slope match it once AllenSDK's "good" label is applied. In dandiset 000022, the session whose movie presentations overlap an invalid "stimulus" interval (sub-769319621) and the two sessions without V1 units leave the paper's 23. The pupil area in `filtered_gaze_mapping` is labelled "Pixels ^ 2" but holds values of the order of 0.001 (pooled tertile edges 0.0016 and 0.0026), too small for an area in pixels. Five of the 30 included sessions have no eye tracking.